# Telecom Customer Churn Analysis

**Goal:** predict which customers are likely to cancel service and explain the patterns associated with churn.

This notebook is the readable, interview-friendly walkthrough. The reusable training logic lives in `src/train.py`.

## 1. Imports and paths

We add the project root to Python's path so the notebook can import our own training functions.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.train import load_and_clean_data, train_and_evaluate

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv'
sns.set_theme(style='whitegrid')

## 2. Load and inspect the raw data

Before modeling, confirm the number of rows, view sample records, and look for incorrect data types or missing values.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f'Rows: {len(raw):,}')
print(f'Columns: {len(raw.columns)}')
raw.head()

In [ ]:
raw.info()

`TotalCharges` is read as text because a small number of rows contain blank strings. The cleaning function converts it to a numeric column and drops only those unusable rows. `customerID` is kept for matching predictions back to customers, but it is not used as a model feature.

In [ ]:
features, target, customer_ids = load_and_clean_data(DATA_PATH)
print(f'Usable rows after cleaning: {len(features):,}')
print(f'Churn rate: {target.mean():.1%}')

## 3. Exploratory data analysis

These charts answer basic business questions before we train a model.

In [ ]:
analysis = features.copy()
analysis['Churn'] = target.map({0: 'No', 1: 'Yes'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=analysis, x='Churn', hue='Churn', legend=False, ax=axes[0])
axes[0].set_title('Class Distribution')
sns.boxplot(data=analysis, x='Churn', y='tenure', hue='Churn', legend=False, ax=axes[1])
axes[1].set_title('Customer Tenure by Churn')
plt.tight_layout();

In [ ]:
contract_churn = (
    analysis.assign(churned=(analysis['Churn'] == 'Yes').astype(int))
    .groupby('Contract', observed=True)['churned']
    .mean()
    .sort_values(ascending=False)
    .mul(100)
)
contract_churn.to_frame('churn_rate_percent')

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=contract_churn.index, y=contract_churn.values, hue=contract_churn.index, legend=False)
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract')
plt.ylabel('Churn rate (%)')
plt.tight_layout();

## 4. Train and compare models

The training function uses the same stratified 80/20 split for every model. All preprocessing is inside each scikit-learn pipeline, so categories from the test set cannot leak into training.

In [ ]:
metrics = train_and_evaluate(DATA_PATH)
metrics.style.format({
    'accuracy': '{:.3f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'roc_auc': '{:.3f}',
})

## 5. Interpretation checklist

After running the notebook, be ready to explain:

- Why churn is a **binary classification** problem.
- Why `customerID` should not be a feature.
- Why preprocessing belongs inside a pipeline.
- Why accuracy alone can be misleading when churners are the smaller class.
- The difference between precision and recall in a retention campaign.
- Why ROC-AUC is useful for ranking customers by risk.
- Which variables were strongest in the winning model and what they mean for the business.